# 1. Introduction à Delta Lake
## Principales fonctionnalités de Delta Lake avec l'API DataFrame en Python

- **Transactions ACID** : Delta Lake garantit la fiabilité des opérations de lecture et d’écriture grâce à des transactions atomiques, cohérentes, isolées et durables.
- **Gestion des versions (Time Travel)** : Il est possible de revenir à une version précédente d’une table pour restaurer ou analyser des données historiques.
- **Mise à jour et suppression de données** : Delta Lake permet d’effectuer des opérations `UPDATE`, `DELETE` et `MERGE` directement sur les DataFrames, ce qui facilite la gestion des données.
- **Schéma évolutif** : Le schéma des tables peut être modifié dynamiquement (ajout ou suppression de colonnes) sans perdre les données existantes.
- **Optimisation des performances** : Delta Lake optimise les requêtes grâce à des fichiers de log transactionnels et des opérations comme `OPTIMIZE` pour compacter les fichiers.
- **Gestion des données en streaming et batch** : L’API DataFrame permet de traiter les données en mode batch ou streaming de façon transparente.

## A. Création de Delta Tables

In [0]:
from pyspark.sql.functions import col

taxi_df = spark.read.table("samples.nyctaxi.trips")

display(taxi_df)

In [0]:
spark.sql("DROP TABLE IF EXISTS delta_taxinyc")

taxi_df = (
    taxi_df.select(
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        col("trip_distance"),
        col("fare_amount"),
        col("pickup_zip"),
        col("dropoff_zip")
    )
    .filter(col("trip_distance") >= 1.0)
)

# ecriture du dataframe dans une delta table
taxi_df.write.mode("overwrite").saveAsTable("delta_nyctaxi")

# affichage de la table
display(spark.table("delta_nyctaxi"))


**NOTE** : Delta Lake est le format par defaut pour Databricks donc on à pas besoin d'utiliser la syntax : `taxi_df.write.format("delta").mode("overwrite").saveAsTable("delta_nyctaxi")` 

In [0]:
%sql
-- on peut faire la même chose en SQL
DROP TABLE IF EXISTS delta_taxinyc_sql;

CREATE OR REPLACE TABLE delta_taxinyc_sql
AS
SELECT 
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip
FROM samples.nyctaxi.trips
WHERE trip_distance >= 1.0;

SELECT * FROM delta_taxinyc_sql;

In [0]:
%sql
-- inspection des métadata de la table
DESCRIBE delta_taxinyc_sql;

In [0]:
%sql
-- inspection des métadata de la table
DESCRIBE EXTENDED delta_taxinyc_sql;

In [0]:
%sql
-- inspection de l'historique de la table
DESCRIBE HISTORY delta_taxinyc_sql;

In [0]:
# on peut utiliser la classe DeltaTable pour manipuler la table delta
from delta.tables import DeltaTable

delta_taxi = DeltaTable.forName(spark, "delta_taxinyc_sql")

display(delta_taxi.history())

## C. Opérations de bases sur les tables Delta avec l'API Dataframe

Opérations DML sur les tables DELTA : 
1. Insert
2. Delete

### 1. INSERT

In [0]:
# On créer un dataframe avec de nouvelles lignes
from datetime import datetime
from decimal import Decimal

new_taxi = spark.createDataFrame([
    (datetime(2022, 1, 1, 12, 0, 0), datetime(2022, 1, 1, 12, 10, 0), float(Decimal(1.5)), float(Decimal(10.0)), 10000, 10001),
    (datetime(2024, 3, 6, 14, 0, 0), datetime(2024, 3, 6, 15, 0, 0), float(Decimal(3.5)), float(Decimal(27.0)), 10001, 10002)
], schema=taxi_df.schema)

# On ajoute les nouvelles lignes dans la table delta
new_taxi.write.format("delta").mode("append").saveAsTable("delta_taxinyc_sql")

# On affiche les nouvelles lignes
display(spark.table("delta_taxinyc_sql").filter(col("tpep_pickup_datetime") >= datetime(2022, 1, 1)))

In [0]:
%sql
-- On peut faire la même chose en SQL
INSERT INTO delta_taxinyc_sql
VALUES
('2026-01-01 14:13:00', '2026-01-01 14:28:00', 2.3, 13.2, 10001, 10002),
('2026-02-01 08:20:00', '2026-02-01 08:45:00', 12.0, 35.6, 10001, 10003);

-- On vérifie l'insert
SELECT * FROM delta_taxinyc_sql WHERE tpep_pickup_datetime >= '2026-01-01';

In [0]:
display(delta_taxi.history())

### 2. DELETE

In [0]:
%sql
delete from delta_taxinyc_sql;

In [0]:
display(delta_taxi.history())

In [0]:
delta_taxi.toDF().count()

## D. Time Travel

Le **Time Travel** permet de revenir à une version précédente d'une table Delta pour restaurer ou analyser des données historiques. Grâce à cette fonctionnalité, on peut interroger la table à un instant ou une version donnée, ce qui facilite l'audit, la récupération ou la comparaison des données.

**Time Travel** peut être réaliser en utilisant :
- **Numéro de version** : `Integer` à partir de 0 (ex. `VERSION AS OF 2`)
- **Timestamps** : ISO-8601 (ex. `TIMESTAMP AS OF '2026-02-19T9:40:00.000Z'`)

In [0]:
# on affiche une déscription des opérations réaliser sur la table
display(delta_taxi.history())

In [0]:
%sql
-- on peut faire la même chose avec sql
DESCRIBE HISTORY delta_taxinyc_sql;

In [0]:
# on voit que la dernière version 3 (actuelle) est celle où la table est vide
# on va restaurer la version 2
previous_version = spark.read.option("versionAsOf", 2).table("delta_taxinyc_sql")
display(previous_version)

In [0]:
%sql
SELECT * FROM delta_taxinyc_sql VERSION AS OF 2
WHERE tpep_pickup_datetime >= '2020-01-01'

In [0]:
# même chose avec les timestamp
timestamp_to_restore = '2026-02-19T08:43:24.000+00:00' # récupérer le timestamp de la version 2 dans la commande history
display(
    spark.read.option("timestampAsOf", timestamp_to_restore).table("delta_taxinyc_sql")
)

## E. Récupération

In [0]:
# on va restaurer la version version 2 dans la table delta_nyctaxi_sql
spark.sql("RESTORE TABLE delta_taxinyc_sql TO VERSION AS OF 2")

# on vérifie que la récupération a fonctionné
display(spark.table("delta_taxinyc_sql"))

## F. MERGE

Le **MERGE** (ou "upsert") avec Delta Lake permet de fusionner des données d'une source (par exemple un DataFrame ou une table) dans une table Delta cible. Cette opération combine l'INSERT, l'UPDATE et le DELETE en une seule commande, selon que les lignes correspondent ou non à une condition de correspondance.

La syntaxe générale est :

```SQL
MERGE INTO table_cible
USING source
ON condition_de_correspondance
WHEN MATCHED THEN
  UPDATE SET ...
WHEN NOT MATCHED THEN
  INSERT ...
```

Cette opération est très utile pour synchroniser des données ou gérer des flux de données incrémentaux.

La table qu'on à utiliser jusque la ne se prête pas bien aux opération de merge (pas d'id ou de clé permettant d'identifier une ligne).

On va donc rajouter une colonne id.

In [0]:
# Ajout d'une colonne id incrémentale à la table delta_nyctaxi_sql
from pyspark.sql.functions import monotonically_increasing_id

df = spark.table("delta_taxinyc_sql").withColumn("id", monotonically_increasing_id())

# Réorganise les colonnes pour mettre 'id' en premier
cols = ['id'] + [col for col in df.columns if col != 'id']
df = df.select(cols)

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("delta_taxinyc_sql")

display(df)

In [0]:
%sql
select * from delta_taxinyc_sql order by id desc;


In [0]:
%sql
-- On créer dans un premier temps une vue temporaire avec une ligne qui est mise à jour
CREATE OR REPLACE TEMPORARY VIEW updated_trips AS
SELECT 16544 AS id,
       '2016-01-15T17:45:33.000+00:00' AS tpep_pickup_datetime,
       '2016-01-15T18:01:03.000+00:00' AS tpep_dropoff_datetime,
       2.9 AS trip_distance, -- on passe de 2.7 a 2.9
       13.5 AS fare_amount, -- on passe de 11.5 a 13.5
       10002 AS pickup_zip,
       11201 AS dropoff_zip
UNION ALL
-- On ajoute une ligne qui n'existe pas dans la table
SELECT 16549 AS id,
       '2026-02-19 10:44:00' AS tpep_pickup_datetime,
       '2020-02-19 11:05:00' AS tpep_dropoff_datetime,
       2.8 AS trip_distance,
       12.4 AS fare_amount,
       10000 AS pickup_zip,
       10000 AS dropoff_zip;


-- on fait l'opération de merge
MERGE INTO delta_taxinyc_sql AS t
USING updated_trips AS s
ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET
    trip_distance = s.trip_distance,
    fare_amount = s.fare_amount
WHEN NOT MATCHED THEN 
  INSERT (id, tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance, fare_amount, pickup_zip, dropoff_zip)
  VALUES (s.id, s.tpep_pickup_datetime, s.tpep_dropoff_datetime, s.trip_distance, s.fare_amount, s.pickup_zip, s.dropoff_zip);
    
-- Vérification du résultat
SELECT * FROM delta_taxinyc_sql WHERE id IN (16544, 16549);


In [0]:
%sql
DESCRIBE HISTORY delta_taxinyc_sql;

## G. Optimisation de la taille des fichiers avec OPTIMIZE

Delta Lake génère de nombreux fichiers au fil du temps car chaque opération d'écriture (INSERT, UPDATE, DELETE, MERGE, etc.) crée de nouveaux fichiers de données ou de journaux (transaction logs) dans le dossier de la table. Cela permet de garantir l'atomicité et la traçabilité des transactions, mais peut entraîner une fragmentation et une multiplication des petits fichiers ("small files") à mesure que les opérations s'accumulent. Cette accumulation peut ralentir les performances de lecture et augmenter les coûts de stockage, d'où l'intérêt d'utiliser des opérations comme `OPTIMIZE` pour compacter les fichiers.

In [0]:
%sql
-- on regarde la colonne numFiles
DESCRIBE DETAIL delta_taxinyc_sql;

In [0]:
%sql
OPTIMIZE delta_taxinyc_sql
ZORDER BY (id, tpep_pickup_datetime);

-- on vérifie les métriques
DESCRIBE HISTORY delta_taxinyc_sql;

In [0]:
%sql
-- le nombre de numFiles devrait avoir diminué (dans mon cas 3 -> 1)
DESCRIBE DETAIL delta_taxinyc_sql;

## H. Suppréssion de versions avec VACUUM

La fonction **VACUUM** dans Delta Lake permet de supprimer définitivement les anciens fichiers de données qui ne sont plus nécessaires, libérant ainsi de l'espace de stockage. Lorsqu'une table Delta est modifiée (INSERT, UPDATE, DELETE, MERGE), les anciennes versions des fichiers sont conservées pour permettre le Time Travel et la récupération. Cependant, ces fichiers deviennent inutiles après un certain temps.

`VACUUM` supprime ces fichiers obsolètes selon une période de rétention configurable (par défaut 7 jours), ce qui optimise l'utilisation du stockage et améliore les performances. Attention : après un VACUUM, il n'est plus possible de revenir à une version antérieure supprimée.

In [0]:
%sql
-- Attention cette commande ne fonctionne pas en serverless
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

-- Supprime les fichier qui ne sont plus référencer par la table Delta et qui sont plus vieux que la période de rétention
-- on vérifie ce qui serait supprimer (sans supprimer en utilisant DRY RUN)
VACUUM delta_taxinyc_sql RETAIN 24 HOURS DRY RUN;

In [0]:
%sql
VACUUM delta_taxinyc_sql RETAIN 24 HOURS;

In [0]:
%sql
DESCRIBE HISTORY delta_taxinyc_sql;

Databricks VACUUM automatiquement par défaut : 
- la période de rétention est de 7 jours (168 heures)
- les tables delta sont *vacuumed* à cette période de rétention
- Check de sécurité pour éviter les erreurs
